In [ ]:
## Packages Import
%matplotlib widget
import copy
import sys
import numpy             as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.ticker import MaxNLocator

sys.path.append("./src/")
from VLMSurface import VLMSurface
from VLMSolver import VLMSolver

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['savefig.format'] = 'pdf'  # Put PDF as default save format
plt.rcParams['savefig.bbox'] = 'tight'  # Cut the figure to the content

In [ ]:
## Utilities

def chord_fn(n, ar, b, sym, space, shape, lam=1.8):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape.

    Input:
        n     -> number of spanwise stations
        ar    -> aspect ratio
        b     -> wing span
        sym   -> symmetric or not
        space -> uniform or cosine spanwise spacing
        shape -> wing planform shape, can be "rectangular", "elliptical" or "tapered"
        lam   -> taper ratio, only needed for tapered shape

    Output:
        c     -> chord length distribution along spanwise direction, shape (n,)
    """
    le = (0.5 * b)
    s_min, s_max = 0.0, le
    if space :
        if sym :
            theta = np.linspace(np.pi/2, np.pi, n)
            s = le*(-np.cos(theta))
        else :
            theta = np.linspace(0, np.pi, n)
            s = le*(1-np.cos(theta))/2
    else :
        s = np.linspace(s_min, s_max, n)
    match shape:
        case "rectangular":
            c = b/ar * np.ones(n)
        case "elliptical":
            c = 4*b/(np.pi*ar) * np.sqrt(1 - (s/le)**2)
        case "tapered":
            c = - s*4*(lam - 1)/(ar*(lam + 1)) + 4*le*lam/(ar*(lam + 1))
    return c

def plot3D(surfaces, show_mirror=False):
    """ 
    Plot all the surface in 3D

    Input
        surfaces    -> a list of VLMSurface
        show_mirror -> enable the display of boundary conditions 
    """
    fig_3d = plt.figure(figsize=(14, 8),constrained_layout=True)
    ax4 = fig_3d.add_subplot(111, projection='3d')
    first = {"wing":True, "wake":True, "mir_wing":True, "mir_wake":True}             # to only have one legend per item
    for surface in surfaces:
        for i, panel in enumerate(surface.wing_panels["real"]):
                pnt    = copy.copy(panel.pnt)
                vrt    = copy.copy(panel.vrt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                vrt.append(vrt[0])
                vrt_plt = np.array(vrt)
                
                #ax4.quiver(panel.ctr[1], panel.ctr[0], panel.ctr[2], panel.normal[1], panel.normal[0], panel.normal[2], length=np.linalg.norm(panel.chord), color='blue')

                if first["wing"] :               # In order to have just one legend
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='black', label='Wing')
                        ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='red', lw=0.8, label='Vortex rings')
                        first["wing"] = False
                else :
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='black')
                        ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='red', lw=0.8)
        for i, panel in enumerate(surface.wake_panels["real"]):
            pnt    = copy.copy(panel.pnt)
            # Close the polygon shape
            pnt.append(pnt[0])
            pnt_plt = np.array(pnt)
            if first["wake"]:               # In order to have just one legend
                    ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='blue', lw=0.6, label='Wake')
                    first["wake"] = False
            else :
                    ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='blue', lw=0.6)
        if show_mirror:
            for i, panel in enumerate(surface.wing_panels["mirror"]):
                    pnt    = copy.copy(panel.pnt)
                    vrt    = copy.copy(panel.vrt)
                    # Close the polygon shape
                    pnt.append(pnt[0])
                    pnt_plt = np.array(pnt)
                    vrt.append(vrt[0])
                    vrt_plt = np.array(vrt)
                    if first["mir_wing"] :               # In order to have just one legend
                            ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='dimgray', label='Mirrored wing')
                            ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='forestgreen', lw=0.8, label='Mirrored vortex rings')
                            first["mir_wing"] = False
                    else :
                            ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='dimgray')
                            ax4.plot(vrt_plt[:, 1], vrt_plt[:, 0], vrt_plt[:, 2], color='forestgreen', lw=0.8)
            for i, panel in enumerate(surface.wake_panels["mirror"]):
                pnt    = copy.copy(panel.pnt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                if first["mir_wake"]:               # In order to have just one legend
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='darkviolet', lw=0.6, label='Mirrored wake')
                        first["mir_wake"] = False
                else :
                        ax4.plot(pnt_plt[:, 1], pnt_plt[:, 0], pnt_plt[:, 2], color='darkviolet', lw=0.6)
            
    ax4.view_init(elev=30, azim=40)
    x_lim = ax4.get_xlim3d()
    y_lim = ax4.get_ylim3d()
    z_lim = ax4.get_zlim3d()
    ax4.set_xlim(x_lim[0],x_lim[1])
    ax4.set_ylim(y_lim[0],y_lim[1]) 
    ax4.set_zlim(z_lim[1],z_lim[0])
    ax4.set_box_aspect([-x_lim[0]+x_lim[1],-y_lim[0]+y_lim[1],-z_lim[0]+z_lim[1]])
    ax4.set_xlabel("y", fontstyle='italic')
    ax4.set_ylabel("x", fontstyle='italic')
    ax4.set_zlabel("z", fontstyle='italic')
    ax4.xaxis.set_major_locator(MultipleLocator(0.5))
    ax4.yaxis.set_major_locator(MultipleLocator(1))
    ax4.zaxis.set_major_locator(MultipleLocator(0.5))

    ax4.set_title('3D view')
    ax4.legend()

    
def plot2D(surfaces, show_mirror, axis):
    """ 
    Plot all the surfaces in 2D

    Input
        surfaces    -> a list of VLMSurface
        show_mirror -> enable the display of boundary conditions 
        axis        -> axis that goes out - 0 for x ; 1 for y ; 2 for z
    """
    fig_2d = plt.figure(figsize=(14/1.5, 8/1.5),constrained_layout=True)
    ax4 = fig_2d.add_subplot()
    first = {"wing":True, "wake":True, "mir_wing":True, "mir_wake":True}             # to only have one legend per item
    for surface in surfaces:
        for i, panel in enumerate(surface.wing_panels["real"]):
                pnt    = copy.copy(panel.pnt)
                vrt    = copy.copy(panel.vrt)
                ctr    = copy.copy(panel.ctr)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                vrt.append(vrt[0])
                vrt_plt = np.array(vrt)
                if first["wing"] :               # In order to have just one legend
                        ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='black', label='Wing panels')
                        ax4.plot(vrt_plt[:, (axis+1)%3], vrt_plt[:, (axis+2)%3], color='red', lw=0.8, label='Vortex rings')
                        #ax4.plot(ctr[(axis+1)%3], ctr[(axis+2)%3], "o", ms=5, markerfacecolor='None', markeredgecolor='red',  label='Control points')
                        first["wing"] = False
                else :
                        ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='black')
                        ax4.plot(vrt_plt[:, (axis+1)%3], vrt_plt[:, (axis+2)%3], color='red', lw=0.8)
                        #ax4.plot(ctr[(axis+1)%3], ctr[(axis+2)%3], "o", ms=5, markerfacecolor='None', markeredgecolor='red')
        for i, panel in enumerate(surface.wake_panels["real"]):
            pnt    = copy.copy(panel.pnt)
            # Close the polygon shape
            pnt.append(pnt[0])
            pnt_plt = np.array(pnt)
            if first["wake"]:               # In order to have just one legend
                    ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='blue', lw=0.6, label='Wake panels')
                    first["wake"] = False
            else :
                    ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='blue', lw=0.6)
        if show_mirror:
            for i, panel in enumerate(surface.wing_panels["mirror"]):
                    pnt    = copy.copy(panel.pnt)
                    vrt    = copy.copy(panel.vrt)
                    # Close the polygon shape
                    pnt.append(pnt[0])
                    pnt_plt = np.array(pnt)
                    vrt.append(vrt[0])
                    vrt_plt = np.array(vrt)
                    if first["mir_wing"] :               # In order to have just one legend
                            ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='dimgray', label='Mirrored wing')
                            ax4.plot(vrt_plt[:, (axis+1)%3], vrt_plt[:, (axis+2)%3], color='forestgreen', lw=0.8, label='Mirrored vortex rings')
                            first["mir_wing"] = False
                    else :
                            ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='dimgray')
                            ax4.plot(vrt_plt[:, (axis+1)%3], vrt_plt[:, (axis+2)%3], color='forestgreen', lw=0.8)
            for i, panel in enumerate(surface.wake_panels["mirror"]):
                pnt    = copy.copy(panel.pnt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                if first["mir_wake"]:               # In order to have just one legend
                        ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='darkviolet', lw=0.6, label='Mirrored wake')
                        first["mir_wake"] = False
                else :
                        ax4.plot(pnt_plt[:, (axis+1)%3], pnt_plt[:, (axis+2)%3], color='darkviolet', lw=0.6)
            
    if axis == 0 :
        ax4.invert_yaxis()
        ax4.set_xlabel(r'$y$' + " [m]" )
        ax4.set_ylabel(r'$z$' + " [m]")
        ax4.set_title('Back view')
    elif axis == 1 :
        for line in ax4.lines:
            xdata, ydata = line.get_xdata(), line.get_ydata()
            line.set_data(ydata, xdata)
        ax4.relim()
        ax4.invert_yaxis()
        ax4.set_xlabel(r'$x$' + " [m]")
        ax4.set_ylabel(r'$z$' + " [m]")
        ax4.set_title('Lateral view')
    else :
        for line in ax4.lines:
            xdata, ydata = line.get_xdata(), line.get_ydata()
            line.set_data(ydata, xdata)
        ax4.set_xlabel(r'$y$' + " [m]")
        ax4.set_ylabel(r'$x$' + " [m]")
        ax4.relim()
        ax4.invert_yaxis()
        ax4.set_title('Top view')

    x_lim = ax4.get_xlim()
    y_lim = ax4.get_ylim()
    ax4.set_xlim(x_lim[0],x_lim[1])
    ax4.set_ylim(y_lim[0],y_lim[1]) 
    ax4.set_aspect('equal')
    
    ax4.grid(True)
    ax4.legend()
    # =======================================================
    # FIN DE LA FONCTION : Placement robuste sans collision
    # =======================================================
    # 'adjustable=box' est crucial pour garder tes chiffres et ton échelle 100% exacts
    ax4.set_aspect('equal', adjustable='box') 
    
    ax4.xaxis.set_major_locator(MultipleLocator(0.1))
    ax4.yaxis.set_major_locator(MultipleLocator(0.1))

    # On force le titre à monter légèrement pour ne jamais être collé au cadre
    ax4.set_title(ax4.get_title(), y=1.08)

    # 1. On place la légende à l'extérieur droit de la figure
    # On utilise loc="center left" et on la décale à X = 1.05 (juste après la fin du graphique)
    ax4.legend(
        loc="center left", 
        bbox_to_anchor=(1.05, 0.5), # Centrée verticalement à droite
        borderaxespad=0,
        ncol=1 # En colonne simple pour rester propre à droite
    )

    # 2. On ajuste manuellement les marges de la figure globale (14, 8)
    # On laisse 25% d'espace vide à droite (right=0.75) pour que la légende y respire 
    # sans jamais compresser le graphique ou toucher le titre.
    fig_2d.subplots_adjust(left=0.08, right=0.75, top=0.85, bottom=0.15)
    ax4.xaxis.set_major_locator(MaxNLocator(nbins=8, prune='both'))
    ax4.yaxis.set_major_locator(MaxNLocator(nbins=6, prune='both'))

def load_csv(filepath):
    """
    Lit le fichier depuis le disque, remplace les virgules par des points 
    et sépare les colonnes X et Y.
    """
    x_list = []
    y_list = []
    
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                # On ignore les lignes vides
                line = line.strip()
                if not line:
                    continue
                
                # Nettoyage : virgule décimale -> point
                clean_line = line.replace(',', '.')
                
                # Séparation par le point-virgule (vu dans ton exemple précédent)
                if ';' in clean_line:
                    parts = clean_line.split(';')
                    x_list.append(float(parts[0].strip()))
                    y_list.append(float(parts[1].strip()))
        
        return np.array(x_list), np.array(y_list)
        
    except FileNotFoundError:
        print(f"Erreur : Le fichier est introuvable à l'adresse : {filepath}")
        return None, None
    except Exception as e:
        print(f"Erreur lors du traitement : {e}")
        return None, None


def plot_ventilation(surface):
    """ 
    Plot the distributions of the ventilation parameters along the spanwise direction 
    
    """
    fig = plt.figure(figsize=(7/1, 4/1),constrained_layout=True)
    ax  = fig.add_subplot()

    x, y = load_csv("data/cavity.csv")
    sorted_ind = np.argsort(x)
    x, y_bottom = x[sorted_ind], y[sorted_ind]
    y_top = np.zeros_like(y_bottom)
    ax.fill_between(x, y_top, y_bottom, 
                facecolor='#90EE90',  # Vert clair (LightGreen)
                alpha=0.6, 
                edgecolor='none')
    ax.plot(x, y_bottom, lw=0.8, color='black', label=r"$L_c/c$ experimental")

    x, y = load_csv("data/Lc.csv")
    sorted_ind = np.argsort(x)
    x, y = x[sorted_ind], y[sorted_ind]
    ax.plot(x, y, "--", color="gray", label=r'$L_c/c$'+"  LLM")
    x, y = load_csv("data/cl.csv")
    sorted_ind = np.argsort(y)
    x, y = x[sorted_ind], y[sorted_ind]
    ax.plot(x, y, "--", color="black", label=r'$C_{l_\text{2D}}$'+"  LLM")
    x, y = load_csv("data/alpha.csv")
    sorted_ind = np.argsort(y)
    x, y = x[sorted_ind], y[sorted_ind]
    ax.plot(x, y, "--", color="red", label=r'$\alpha_{\text{eff}}/\alpha$'+"  LLM")
    x, y = load_csv("data/a0.csv")
    sorted_ind = np.argsort(y)
    x, y = x[sorted_ind], y[sorted_ind]
    ax.plot(x, y, "--", color="magenta", label=r'$a_0/(2\pi)$'+"  LLM")
    x, y = load_csv("data/sigma.csv")
    sorted_ind = np.argsort(y)
    x, y = x[sorted_ind], y[sorted_ind]
    ax.plot(x, y, "--", color="blue", label=r'$\sigma_c$')



    ctrl    = np.array([p.ctr for p in surface.wing_panels["real"]])
    z       = np.sum(ctrl.reshape((M,2*N,3)), axis=0)[:,2]/M
    z       = z/B
    vent    = surface.ventilation

    ax.plot(vent["Lc/c"], z, color="gray", label=r'$L_c/c$'+"  VLM")
    ax.plot(vent["a0"]*np.sin(vent["alpha_eff"]),z, color="black", label=r'$C_{l_\text{2D}}$'+"  VLM")
    ax.plot(vent["alpha_eff"]/(np.pi*BETA/180), z, color="red", label=r'$\alpha_{\text{eff}}/\alpha$'+"  VLM")
    ax.plot(vent["a0"]/(2*np.pi), z, color="magenta", label=r'$a_0/(2\pi)$'+"  VLM")
    ax.plot(vent["sigma_c"], z, color="blue")
    ax.plot()
    ax.set_xlim(-0.2, 1.9)
    ax.set_ylim(1.2,-0.3)
    ax.xaxis.set_major_locator(MultipleLocator(0.2))
    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.set_xlabel(r'$x/c$')
    ax.set_ylabel(r'$z/h$')
    #ax.grid()
    ax.legend(loc="lower right")

In [ ]:
## Parameters Definition

# Wing geometry
B = 0.0453   # wing span                 [m]
AR = 1        # aspect ration             [-]
ALPHA  = 0     # angle of attack           [deg] - positive definite for counterclockwise rotations about y-axis
BETA   = 10     # drift angle               [deg] - positive definite for counterclockwise rotations about z-axis
LAMBDA = 0      # middle-chord sweep angle  [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
DELTA  = 0      # dihedral angle            [deg] - positive definite for rotations oriented towards positive y-axis  X-AXIS, négatif pour dyhedre classique 
PHI    = 0      # twist tip angle           [deg] - negative for washout

SYM   = True          # symmetric wing configuration
SPACE = False  # spacing distribution, False = uniform, True = cos
SHAPE = "rectangular"   # shape of the wing
TIME  = "classic"       # time step distribution - classic or cosine at the starting vortex
FREE  = "antisymmetric"     # free surface boundary condition - symmetric (wall) ; antisymmetric (zero delta pressure) ; None

FROUDE = 1.5

# Flow properties
U = 1#FROUDE*np.sqrt(B*9.81)     # inflow velocity [m/s]

# Wing discretization
N = 10    # number of panels in spanwise direction
M = 10    # number of panels in chordwise direction

# Time simulation parameters
T  =  8*B/U/AR # length of the simulation in s
DT = T/50      # time step lentgh 




In [ ]:
## Global Ventilation 

def Ventilation(vlm):
    surfaces    = vlm.surfaces
    u           = np.linalg.norm(vlm.U)
    n           = []                     # number of spanwise panel of each surface
    correction  = []    
    Cl          = []
    slope       = []              
    # Initialization :
    vlm._time_sim()
    vlm._kuttas_loads()
    for i,surface in enumerate(surfaces):
        n.append(surface.N)
        correction.append(np.zeros(n[i]))
        orientation=(1)                                                 # orientation of the normals in comparison to the flow / 1 for upward -1 for downward
        Cl.append(surface.Cl_2d)
        # getting the lift coefficient slope to scale the correction
        CL = 2*np.linalg.norm(surface.loads[1:])/(surface.area*u**2)    # global lift coef
        mean_normal = np.mean(surface.normals, axis=0)
        mean_normal_unit = mean_normal / np.linalg.norm(mean_normal)
        cos_avg = mean_normal_unit[0] 
        if cos_avg < 0 :
            orientation = -1
        surface.orientation = orientation
        avg_aoa = np.abs(np.arcsin(cos_avg))            # average angle of attack
        slope.append(CL/avg_aoa) 
    #correction, Cl, slope = np.array([correction]), np.array([Cl]), np.array([slope])
    a0        = 2*np.pi*np.ones_like(Cl)
    delta_aoa = copy.copy(a0)
    k         = 0
    aoa       = copy.copy(a0)            # just for the first test
    while k<400 and np.max(np.abs(a0*aoa - Cl)) > 1e-2: # np.linalg.norm(a0[0]*aoa[0] - Cl[0])
        if k == 0 :                     # to ensure that the non ventilated surfaces passes the test
            a0        = np.copy(Cl)
            aoa       = np.ones_like(a0)         
        k += 1
        for i,surface in enumerate(surfaces):
            if surface.ventilate :
                aoa[i] = (Cl[i]/(2*np.pi))-correction[i]
                surface._ventilation_coef(aoa[i].reshape(-1),u)
                a0[i] = copy.copy(surface.ventilation["a0"])
                delta_aoa[i] = 0.1*(a0[i]*(aoa[i]) - Cl[i])/(slope[i])
                correction[i] = correction[i] + delta_aoa[i]        
        vlm._correct(correction)
        vlm._time_sim()
        vlm._kuttas_loads()
        for i,surface in enumerate(surfaces):
            if surface.ventilate :
                Cl[i] = surface.Cl_2d
    if k==400:
        print("Did not converge")
    for i,surface in enumerate(surfaces):
            if surface.ventilate :
                surface._ventilation_coef(aoa[i].reshape(-1),u)
  




In [ ]:
## test Global ventilation
surface3 = VLMSurface(origin=np.array([0,0,0.5*B]),
                        plan=0,
                        boundary=FREE,
                        shedding="right",
                        b=B,
                        c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                        alpha=np.deg2rad(ALPHA),
                        beta=np.deg2rad(BETA),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=np.deg2rad(PHI),
                        sym=SYM,
                        space=SPACE,
                        n=N,
                        m=M,
                        vent=True)
vlm = VLMSolver([surface3], np.array([U,0,0]),FREE)
vlm._ventilation()
print("Cl = ", 2*np.linalg.norm(surface3.loads[1:])/(surface3.area*U**2))
plot3D([surface3])
plot_ventilation(surface3)


In [ ]:
## Ventilation V4



def iterative_vent(beta,plot):
    BETA = beta
    surface3 = VLMSurface(origin=np.array([0,0,0.5*B]),
                        plan=0,
                        boundary=FREE,
                        shedding="right",
                        b=B,
                        c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                        alpha=np.deg2rad(ALPHA),
                        beta=np.deg2rad(BETA),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=np.deg2rad(PHI),
                        sym=SYM,
                        space=SPACE,
                        n=N,
                        m=M,
                        vent=True)
    da = np.zeros(2*N)
    vlm = VLMSolver([surface3], np.array([U,0,0]), FREE)
    vlm._time_sim(t=B*T/(AR*0.0453), dt=B*DT/(AR*0.0453), distribution=TIME)
    #plot3D([surface3],False)
    vlm._kuttas_loads()
    Cl = copy.copy(surface3.Cl_2d)
    print("Cl :",Cl)
    slope1 = 2*surface3.loads[1]*AR/(B**2)/(U**2)/np.deg2rad(BETA)
    print("slope",slope1)
    slope = copy.copy(surface3.Cl_2d)/np.deg2rad(BETA)
    a0 = 2*np.pi*np.ones_like(Cl)
    phi = np.zeros(2*N+1)
    k = 00
    aoa = np.pi*np.ones(2*N)            # just for the fisrst test
    while k<400 and np.linalg.norm(a0*(aoa) - Cl)>1e-2:          
        k += 1
        aoa = (Cl/(2*np.pi))-da
        surface3._ventilation_coef(aoa,1)
        a0 = copy.copy(surface3.ventilation["a0"])
        delta_aoa = 0.1*(a0*(aoa) - Cl)/(slope1)
        da = da + delta_aoa
        #phi = phi + np.concatenate([np.zeros(1),(delta_aoa[1:] + delta_aoa[:-1])/2, 0.5*np.array([delta_aoa[-1]])])
        surface3 = VLMSurface(origin=np.array([0,0,0.5*B]),
                        plan=0,
                        boundary=FREE,
                        shedding="right",
                        b=B,
                        c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                        alpha=np.deg2rad(ALPHA),
                        beta=np.deg2rad(BETA),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=phi,
                        sym=SYM,
                        space=SPACE,
                        n=N,
                        m=M,
                        vent=True)
        vlm = VLMSolver([surface3], np.array([U,0,0]), FREE)
        vlm._correct([da])
        vlm._time_sim(t=B*T/(AR*0.0453), dt=B*DT/(AR*0.0453), distribution=TIME)
        vlm._kuttas_loads()
        if k == 1:
            print("Correction [°]:", da) 
            print("a0 :", a0)
            print("normals :", vlm.normals[:20])
        Cl = copy.copy(surface3.Cl_2d)
    if k==400:
        print("Did not converge")
    if plot :
        surface3._ventilation_coef(aoa,1)
        plot3D([surface3],False)
        plot_ventilation(surface3)
    print("T  :",B*T/(AR*0.0453))
    print("DT :",B*DT/(AR*0.0453))
    return 2*surface3.loads*AR/(B**2)/(U**2)

print("Cl : ", iterative_vent(BETA,True)[1])







In [ ]:
# random test

surface = VLMSurface(origin=np.array([0,0,0]),
                        plan=1,
                        boundary=FREE,
                        shedding="both",
                        b=B,
                        c=chord_fn((2*N+1), 4, B, False, SPACE, SHAPE),
                        alpha=-np.deg2rad(10),
                        beta=np.deg2rad(0),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=np.deg2rad(PHI),
                        sym=False,
                        space=SPACE,
                        n=2*N,
                        m=M,
                        vent=False)
surface._set_normals(0)
orientation=(1)                                                 # orientation of the normals in comparison to the flow / 1 for upward -1 for downward
mean_normal = np.mean(surface.normals, axis=0)
mean_normal_unit = mean_normal / np.linalg.norm(mean_normal)
cos_avg = mean_normal_unit[0] 
avg_aoa = np.abs(np.arcsin(cos_avg))            # average angle of attack
print("original aoa",np.rad2deg(avg_aoa))
if cos_avg < 0 :
    orientation = -1
surface.orientation = orientation
correction2 = np.linspace(np.deg2rad(10), np.deg2rad(10),2*N)
vlm = VLMSolver([surface], np.array([U,0,0]), FREE)
vlm._correct([correction2])
# vlm._time_sim(T, DT, TIME)
# vlm._kuttas_loads()
plot3D([surface], False)
mean_normal = np.mean(surface.normals, axis=0)
mean_normal_unit = mean_normal / np.linalg.norm(mean_normal)
cos_avg = mean_normal_unit[0] 
avg_aoa = np.abs(np.arcsin(cos_avg))            # average angle of attack
print("estimated aoa",np.rad2deg(avg_aoa))
#print("CL           ",2*np.linalg.norm(surface.loads[1:])/(surface.area*U**2) )
# fig = plt.figure(figsize=(14/1.5, 8/1.5),constrained_layout=True)
# ax  = fig.add_subplot()
# ctrl     = np.array([p.ctr for p in surface.wing_panels["real"]])
# z        = np.sum(ctrl.reshape((M,2*N,3)), axis=0)[:,2]/M 
# ax.plot(z, surface.Cl_2d, label="new")
# Cl_old= [0.38032247, 0.36651316, 0.36249562, 0.36687121, 0.37359898, 0.38044368, 0.38645063, 0.39117703, 0.39440904, 0.39604532, 0.39604532, 0.3944090, 0.39117703 ,0.38645063, 0.38044368, 0.37359898, 0.36687121 ,0.36249562, 0.36651316, 0.38032247]
# ax.plot(z, Cl_old, "--", label="target")
# ax.plot(z, surface.Cl_old, ".-", label="old")
# ax.set_xlabel("Spanwise location "+ r"$y/b$")
# ax.set_ylabel("2D lift coefficient "+ r"$C_l$")
# ax.set_ylim(np.min(np.concatenate([surface.Cl_2d, np.array([0])])), np.max(surface.Cl_2d)*1.1)
# ax.legend()
# ax.grid()
# print("VLM lift coef           :", 2*surface.loads[1]*AR/(B**2), "(AR=",AR,"; ALPHA=",ALPHA,"°)" )

In [ ]:
## Class A simulation


def classA_sim(both, fly_regime, phi, plot):
    """ 
    Input :
        both       -> True if the two foils are simulated, False if only one the port one is simulated
        fly_regime -> Index of the flying height regime (equal to 10*flying height)
        phi        -> Array of correction angles for ventilation for each board, in rad
        plot       -> True if the results should be plotted
    Output :
        vlm        -> VLMSolver object containing the results of the simulation
        h          -> Total immersioon depth 
    """
    flying_height   = np.array([0, 0.1, 0.2, 0.3, 0.4, 0.5])
    Horizontal_lift = np.array([61, 63, 65, 68, 72, 79])
    AoA             = np.array([6.8, 6.87, 6.92, 6.75, 6.85, 7.12])        
    Leeway_angle    = np.array([1.22, 1.53, 2.02, 2.86, 4.55, 9.2])
    BETA            = Leeway_angle[fly_regime]
    ALPHA           = AoA[fly_regime]
    B_MAST     = 1.46
    FLYING_HEIGHT = flying_height[fly_regime]
    DELTA = -10.3
    LAMBDA = 0      # middle-chord sweep angle  [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
    PHI    = 0      # twist tip angle           [deg] - negative for washout

    SPACE = True  # spacing distribution, False = uniform, True = cos
    TIME  = "classic"       # time step distribution - classic or cosine at the starting vortex
    FREE  = "antisymmetric"     # free surface boundary condition - symmetric (wall) ; antisymmetric (zero delta pressure) ; None
    
    # Flow properties
    U = 15*1.852/3.6     # inflow velocity [m/s]

    # Wing discretization
    N = 10    # number of panels in spanwise direction
    M = 10    # number of panels in chordwise direction

    # Time simulation parameters
    T  =  15*0.16/U # length of the simulation in s
    DT = T/80      # time step lentgh 



    surface2 = VLMSurface(origin=np.array([0,-1,0]),
                            plan=0,
                            boundary=FREE,
                            shedding="none",
                            b=B_MAST-2*FLYING_HEIGHT/np.cos(np.deg2rad(DELTA)),
                            c=chord_fn((N+1), B_MAST/0.16, B_MAST, False, SPACE, "rectangular"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(DELTA),
                            phi=phi[0],
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True)
    B = 0.8
    surface3 = VLMSurface(origin=np.array([0,-1+B/2 + (B_MAST-2*FLYING_HEIGHT/np.cos(np.deg2rad(DELTA)))*np.sin(np.deg2rad(DELTA))/2, ((B_MAST*np.cos(np.deg2rad(DELTA))-2*FLYING_HEIGHT))/2]),
                            plan=1,
                            boundary=FREE,
                            shedding="left",
                            b=B,
                            c=chord_fn((N+1), B/0.16, B, False, SPACE, "rectangular"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(0),
                            phi=-phi[1],
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True)

    surface4 = VLMSurface(origin=np.array([0,1,0]),
                            plan=0,
                            boundary=FREE,
                            shedding="none",
                            b=B_MAST-2*FLYING_HEIGHT/np.cos(np.deg2rad(DELTA)),
                            c=chord_fn((N+1), B_MAST/0.16, B_MAST, False, SPACE, "rectangular"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(-DELTA),
                            phi=phi[0],
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M,
                            vent=True)

    surface5 = VLMSurface(origin=np.array([0, 1-(B_MAST-2*FLYING_HEIGHT/np.cos(np.deg2rad(DELTA)))*np.sin(np.deg2rad(DELTA))/2, ((B_MAST*np.cos(np.deg2rad(DELTA))-2*FLYING_HEIGHT))/2]),
                            plan=1,
                            boundary=FREE,
                            shedding="right",
                            b=B,
                            c=chord_fn((N+1), B/0.16, B, False, SPACE, "rectangular"),
                            alpha=np.deg2rad(ALPHA),
                            beta=np.deg2rad(BETA),
                            lamb=np.deg2rad(LAMBDA),
                            delta=np.deg2rad(0),
                            phi=-phi[1],
                            sym=False,
                            space=SPACE,
                            n=N,
                            m=M, 
                            vent=True)

    if both :
        surfaces = [surface2, surface3, surface4, surface5]
    else :
        surfaces = [surface2, surface3]
    if plot :
        plot2D(surfaces,False, 0)
    vlm = VLMSolver(surfaces, np.array([U,0,0]), FREE)
    # vlm._time_sim(t=T, dt=DT, distribution=TIME)
    # vlm._kuttas_loads()
    #L_tot = 1025*np.sum([surface.loads for surface in surfaces], axis=0)/9.81
    
    # print("Horizontal Lift : ", L_tot[1])
    # print("Vertical Lift : ", -L_tot[2]) 
    if plot : 
        
        plot3D(surfaces,False)
    
    return vlm, ((B_MAST*np.cos(np.deg2rad(DELTA))-2*FLYING_HEIGHT))/2



# vlm = classA_sim(both=False, fly_regime=4, phi=np.array([0,0,0,0]), plot=True)[0]
# print("Cl = ",vlm.surfaces[0].Cl_2d)
#L_tot = classA_vent(fly_regime=4, ventilate=2, plot=True)
# vent = [1, 2]
# flyght_reg = [0, 1, 2, 3, 4, 5]
# F_vert=[]
# F_hor=[]
# for f in flyght_reg :
#     vlm = classA_sim(both=True, fly_regime=f, phi=np.array([0,0,0,0]), plot=True)[0]
#     L_tot = 1025*np.sum([surface.loads for surface in vlm.surfaces], axis=0)/9.81
#     F_vert.append(-L_tot[2])
#     F_hor.append(L_tot[1])
# print("Vertical Lift = ", F_vert)
# print("Horizontal Lift = ", F_hor)

In [ ]:

vlm = classA_sim(False,3,np.array([0,0,0,0]),True)[0]
vlm._ventilation()
L_tot = 1025*np.sum([surface.loads for surface in vlm.surfaces], axis=0)/9.81
print("Vertical Lift = ", -L_tot[2])
print("Horizontal Lift = ", L_tot[1])

In [ ]:
## Ventilation class A

def classA_vent(fly_regime, ventilate, plot):
    da = np.tile(np.zeros(N), (ventilate,1))
    vlm, h =classA_sim(False, fly_regime, np.zeros(N+1), False)
    FROUDE = U/(np.sqrt(9.81*h))
    Cl = np.array([surface.Cl_2d for surface in vlm.surfaces])
    n_surf = ventilate
    Cl = np.abs(Cl[:n_surf])
    slope1 = np.abs(np.array([2*surface.loads[surface.plan+1]*AR/(B**2)/(U**2)/np.deg2rad(BETA) for surface in vlm.surfaces]))
    a0 = 2*np.pi*np.ones_like(Cl)
    delta_aoa = copy.copy(a0)
    phi = np.zeros((4, N+1))
    k = 00
    aoa = np.tile(np.pi*np.ones(N),(n_surf,1))            # just for the fisrst test
    while k<400 and np.max(np.abs(a0*aoa - Cl)) > 1e-2:          
        k += 1
        for i in range(ventilate):
            surface = vlm.surfaces[i]
            aoa[i] = (Cl[i]/(2*np.pi))-da[i]
            surface._ventilation_coef(aoa[i],FROUDE)
            a0[i] = copy.copy(surface.ventilation["a0"])
            delta_aoa[i] = 1000*0.05*(a0[i]*(aoa[i]) - Cl[i])/(slope1[i])
            da[i] = da[i] + delta_aoa[i]
            phi[i] = phi[i] + np.concatenate([np.zeros(1),(delta_aoa[i][1:] + delta_aoa[i][:-1])/2, 0*np.array([delta_aoa[i][-1]])])
        vlm = classA_sim(False, fly_regime, phi, False)[0]
        for i in range(ventilate):
            Cl[i] = np.abs(copy.copy(vlm.surfaces[i].Cl_2d))
    if k==400:
        print("Did not converge")
    if plot :
        plot3D(vlm.surfaces,False)
    return 1025*np.sum([surface.loads for surface in vlm.surfaces], axis=0)/9.81



In [ ]:
## Ventilated validation

beta = np.array([2.5,5,7.5,10,12.5,15,17.5,20,22.5,25,27.5,30])
asp = [0.5, 1, 1.5]
Fn  = [[2.5, 4.5],[1.5, 2.5, 3.5],[1.5, 2, 3]]
print("alpha = ", beta)
for i, ar in enumerate(asp) :
    AR = ar
    print("Ar : ", ar)
    for f in Fn[i] :
        FROUDE = f
        B      = 1/(9.81*FROUDE**2)
        cl_kut = []
        cd_kut = []
        for b in beta : 
            c = iterative_vent(b, False)
            cl_kut.append(c[1])
            cd_kut.append(c[0])
        print("Fn : ", f)
        print("cl_kut = ", cl_kut)
        print("cd_kut = ", cd_kut)
        print(" ")

In [ ]:
## Wetted validation

beta = np.linspace(0,12,13)
FREE = "no"
asp = [ 3]
Fn  = [ 1.5]
print("alpha = ", beta)
for ar in asp :
    AR = ar
    print("Ar : ", ar)
    for f in Fn :
        FROUDE = f
        B      = 1/(9.81*FROUDE**2)
        cl_kut = []
        cd_kut = []
        for b in beta : 
            surface3 = VLMSurface(origin=np.array([0,0,0.5*B]),
                        plan=0,
                        boundary=FREE,
                        shedding="both",
                        b=B,
                        c=chord_fn((N+1), AR, B, SYM, SPACE, SHAPE),
                        alpha=np.deg2rad(ALPHA),
                        beta=np.deg2rad(b),
                        lamb=np.deg2rad(LAMBDA),
                        delta=np.deg2rad(DELTA),
                        phi=np.deg2rad(PHI),
                        sym=SYM,
                        space=SPACE,
                        n=N,
                        m=M)
            vlm = VLMSolver([surface3], np.array([U,0,0]), FREE, ventilation=True, correction=np.zeros(2*N))
            vlm._time_sim(t=B*T/(AR*0.0453), dt=B*DT/(AR*0.0453), distribution=TIME)
            vlm._kuttas_loads()
            c = surface3.loads
            cl_kut.append(2*c[1]*AR/(B**2))
            cd_kut.append(2*c[0]*AR/(B**2))
        print("Fn : ", f)
        print("cl_kut = ", cl_kut)
        print("cd_kut = ", cd_kut)
        print(" ")